# La France est-elle en retard en IA, ou simplement plus transparente ?

*Projet Python pour la data science, ENSAE 2A (2026-2027)*

Ce notebook est notre rapport final. Il s'exécute de bout en bout, de la collecte des données aux conclusions. Les analyses complémentaires (tests de robustesse, graphiques supplémentaires) sont dans le dossier `notebooks/`.

## Introduction

On entend souvent que la France est en retard en intelligence artificielle. Elle a pourtant Mistral AI, l'un des seuls développeurs européens de grands modèles de langage, et 18 % des entreprises implantées en France utilisaient au moins une technologie d'IA en 2025 ([Insee](https://www.insee.fr/fr/statistiques/9025878)). Depuis le 2 août 2025, l'AI Act impose aussi des obligations aux développeurs de modèles d'IA à usage général, avec un code de bonnes pratiques que chacun peut signer.

Mais pour mesurer un retard, il faut connaître les modèles des leaders, et ceux-ci publient de moins en moins d'informations. On se pose donc deux questions. À quelle étape la France décroche-t-elle : la recherche, les modèles ou leur diffusion ? Et le manque de transparence des leaders fausse-t-il la comparaison ?

## Démarche

On compare la France à cinq pays (États-Unis, Chine, Royaume-Uni, Allemagne, Canada) à trois étapes : la recherche, les modèles et leur diffusion. On regarde ensuite si les développeurs publient de moins en moins d'informations sur leurs modèles, et si cela change la mesure du retard.

Avant de regarder les données, on a posé cinq hypothèses :

- **H1** : la France publie beaucoup en IA mais produit peu de modèles.
- **H2** : sa présence parmi les modèles repose sur très peu d'acteurs.
- **H3** : les développeurs publient de moins en moins d'informations sur leurs modèles depuis 2020.
- **H4** : les modèles dont les poids sont publiés (modèles ouverts) sont mieux documentés que les modèles fermés.
- **H5** : les signataires du code de bonnes pratiques de l'AI Act sont devenus plus transparents.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy

import geopandas as gpd

from config import FIGURES, URL_FOND_MONDE
from style import COULEURS_PAYS, appliquer_style
from nettoyage import (charger_dernier, preparer_epoch, eclater_pays,
                       annees_de_retard, INFOS_PUBLIABLES)
from texte import nettoyer, comparer_presence, tfidf_par_groupe, nuage
from graphiques import (courbes_par_pays, profil_maillons, barres_rapport,
                        barres_par_pays, nuage_de_points_annote, carte_monde)
import analyses

appliquer_style()
FIGURES.mkdir(exist_ok=True)
pd.set_option("display.precision", 2)

## 1. Les données

### Collecte

Nous avons utilisé six sources, toutes collectées par un script du dossier `src/`. La principale est Epoch AI, qui recense les modèles d'IA notables. Nous nous sommes aussi servis de l'index de transparence de Stanford. Ces deux sources sont des fichiers CSV. Les publications scientifiques en IA viennent de l'API d'OpenAlex, les téléchargements des modèles ouverts de l'API de Hugging Face et la population et le PIB de chaque pays de l'API de la Banque mondiale. Pour finir, nous avons récupéré par web scraping, sur le site de la Commission européenne, la liste des signataires du code de bonnes pratiques de l'AI Act.

Chaque script enregistre une copie datée dans `data/raw/` et une copie au format Parquet est aussi ajoutée sur le stockage MinIO du SSP Cloud. Comme les sources sont mises à jour régulièrement, le rapport relit par défaut ces copies, afin d'obtenir les mêmes résultats à chaque fois. Pour tout re-télécharger, il suffit de passer la variable `COLLECTER` à `True`. La collecte prend quelques minutes et une clé OpenAlex gratuite, placée dans la variable d'environnement `OPENALEX_API_KEY`, permet de ne pas être limité par l'API.

In [ ]:
COLLECTER = False

if COLLECTER:
    from pipeline import collecter_tout
    collecter_tout()

In [ ]:
epoch = preparer_epoch(charger_dernier("epoch_notable_models"))
epoch_pays = eclater_pays(epoch)
publications = charger_dernier("openalex_publications")
resumes = charger_dernier("openalex_resumes")
hf = charger_dernier("hf_models")
fmti = charger_dernier("fmti_scores")
signataires = charger_dernier("signataires_code")
pays_info = charger_dernier("banque_mondiale")

pd.DataFrame({
    "lignes": [len(epoch), len(publications), len(resumes), len(hf), fmti["developpeur"].nunique(), len(signataires), len(pays_info)],
    "unité": ["modèles", "pays × années", "résumés d'articles", "modèles", "développeurs notés", "signataires", "pays × années"],
    "période": [f"{epoch['annee'].min()}-{epoch['annee'].max()}",
                f"{publications['annee'].min()}-{publications['annee'].max()}",
                f"{resumes['annee'].min()}-{resumes['annee'].max()}",
                f"{hf['createdAt'].str[:4].min()}-{hf['createdAt'].str[:4].max()}",
                ", ".join(sorted(fmti["edition"].unique())), "liste à date",
                f"{pays_info['annee'].min()}-{pays_info['annee'].max()}"],
}, index=["Epoch AI", "OpenAlex (comptages)", "OpenAlex (résumés)", "Hugging Face",
         "Stanford FMTI", "Code européen", "Banque mondiale"])

### Choix de nettoyage

Notre projet cherche à répondre à deux questions. La France est-elle en retard en IA ? Et le fait que les grands développeurs publient de moins en moins d'informations fausse-t-il cette comparaison ? Les données brutes ne permettent pas d'y répondre directement, et chacun des choix qui suivent sert l'une de ces deux questions.

Pour mesurer le retard de la France, il faut d'abord compter combien de modèles chaque pays a développés. La difficulté est que certains modèles sont le fruit d'une collaboration entre plusieurs pays. Si on les comptait en entier pour chacun d'eux, un même modèle serait compté plusieurs fois et les pays qui collaborent beaucoup paraîtraient plus productifs qu'ils ne le sont. Nous avons donc partagé chaque modèle entre les pays qui l'ont développé. Ainsi, un modèle conçu par une entreprise américaine et un laboratoire français compte pour moitié aux États-Unis et pour moitié en France. C'est ce qu'on appelle le comptage fractionnaire.

La seconde question porte sur la transparence. Pour comparer la France aux leaders, il faut connaître les caractéristiques de leurs modèles, en particulier leur puissance de calcul, qui indique à quel point un modèle est avancé. Si ces informations manquent, on ne peut pas savoir précisément où en sont les leaders, ni donc mesurer correctement le retard de la France. Pour chaque modèle, nous avons donc relevé si cinq informations étaient connues, à savoir la puissance de calcul, le nombre de paramètres, la taille des données d'entraînement, le coût et le matériel utilisé. Cela nous permet de voir si les développeurs publient de moins en moins, et lesquels.

Pour comprendre pourquoi certains développeurs publient moins que d'autres, nous avons aussi distingué les modèles ouverts, que tout le monde peut télécharger comme ceux de Mistral, des modèles fermés, qu'on ne peut utiliser qu'en ligne comme ChatGPT. Un développeur qui garde son modèle fermé le fait souvent pour des raisons commerciales, et on peut s'attendre à ce qu'il cache aussi davantage d'informations sur la manière dont il l'a construit. Lorsque Epoch ne précise pas si un modèle est ouvert, nous avons laissé l'information vide plutôt que de le supposer fermé.

Enfin, quelques corrections rendent les données plus fiables. Nous avons inclus les articles publiés en conférence, principal lieu de publication de la recherche en IA, pour ne pas sous-estimer la recherche française. Nous avons fait correspondre les noms d'entreprises qui varient d'une base à l'autre, comme « Google » et « Google DeepMind », afin de pouvoir croiser les différentes sources. Nous avons aussi retiré les articles d'informatique quantique, qu'OpenAlex classe à tort dans l'intelligence artificielle.

## 2. Où la France décroche-t-elle ?

### 2.0 Où sont produits les modèles d'IA ?

Nombre de modèles notables par pays depuis 1950 (comptage fractionnaire). La carte utilise la projection Equal Earth, qui respecte les surfaces, et les classes sont choisies à la main.

In [ ]:
fond = gpd.read_file(URL_FOND_MONDE)[["ADMIN", "geometry"]]
carte, absents = analyses.modeles_par_pays_monde(epoch_pays, fond)
carte_monde(carte, "modeles", "Les modèles d'IA notables, pays par pays",
            fichier=FIGURES / "rapport_0_carte.png")
plt.show()
print("Absents de la carte (territoires trop petits à cette échelle, ou non localisés) :",
      absents.round(1).to_dict())
carte.sort_values("modeles", ascending=False)[["ADMIN", "modeles"]].head(10).round(1)

**Interprétation.** Les États-Unis ont produit 644 modèles notables et la Chine 169. Viennent ensuite le Royaume-Uni (72), le Canada (43), la France (29), la Corée du Sud (25) et l'Allemagne (21). La France est parmi les premiers pays européens, mais avec 22 fois moins de modèles que les États-Unis. Hong Kong et Singapour (9 modèles chacun) sont trop petits pour apparaître sur la carte.

### 2.1 La France à chaque étape

Part de chaque pays parmi les six comparés, sur 2024-2025 : articles d'IA, articles dans le 1 % les plus cités, modèles notables, modèles de pointe et téléchargements sur Hugging Face.

In [ ]:
chaine = analyses.chaine_par_periode(publications, epoch_pays)
profil = analyses.profil_recent(chaine, hf)
profil_maillons(profil, "Part de chaque pays, de la recherche à la diffusion (2024-2025)",
                fichier=FIGURES / "rapport_1_profil.png")
plt.show()
profil.style.format("{:.1%}")

**Interprétation.** H1 est confirmée : la part de la France diminue à chaque étape. Sur 2024-2025, elle représente 4,3 % des articles d'IA des six pays, 2,3 % des articles les plus cités, 2,6 % des modèles notables et aucun modèle de pointe (les États-Unis en ont 94 %). Par habitant, la France publie pourtant presque autant que les États-Unis : 171 articles par million d'habitants sur 2022-2024, contre 188 aux États-Unis, 190 en Allemagne, 254 au Canada et 277 au Royaume-Uni. Pour la diffusion, les modèles français représentent 1,9 % des téléchargements sur Hugging Face, loin derrière la Chine (50 %) et les États-Unis (46 %).

### 2.2 De la recherche aux modèles

Nombre de modèles notables pour 1 000 articles d'IA, par période.

In [ ]:
transformation = chaine.pivot(index="pays", columns="periode", values="modeles_pour_1000_articles").reindex(analyses.PAYS)
transformation.round(2)

**Interprétation.** Sur 2024-2025, la France produit 0,6 modèle pour 1 000 articles, contre 2,1 aux États-Unis, et ce ratio bouge peu depuis 2018 (0,66 puis 0,74). La Chine, elle, le triple (de 0,23 à 0,70). La chute du Royaume-Uni (de 1,34 à 0,13) vient en partie du fait qu'Epoch attribue Google DeepMind aux États-Unis depuis la fusion de 2023. Attention : les articles sont comptés en entier et les modèles en fractionnaire, donc ces ratios servent à comparer les pays et les périodes, pas à lire une valeur absolue.

### 2.3 Une présence qui tient à peu d'acteurs

In [ ]:
fr = epoch_pays.loc[epoch_pays["pays"] == "France"]
par_organisation = fr.groupby("Organization")["poids"].sum().sort_values(ascending=False)
depuis_2024 = fr.loc[fr["annee"] >= 2024]
print(f"Modèles français depuis 2024 : {depuis_2024['poids'].sum():.1f}, "
      f"dont Mistral AI : {depuis_2024.loc[depuis_2024['Organization'].str.contains('Mistral', na=False), 'poids'].sum():.1f}")

comptages = pd.DataFrame({
    "fractionnaire": epoch_pays.groupby("pays")["poids"].sum(),
    "entier": eclater_pays(epoch, "entier").groupby("pays")["poids"].sum(),
}).reindex(analyses.PAYS)
comptages["entier / fractionnaire"] = comptages["entier"] / comptages["fractionnaire"]
display(par_organisation.head(5).round(2))
comptages.round(2)

**Interprétation.** H2 est confirmée : depuis 2024, les sept modèles notables français sont tous de Mistral AI. Sur toute la période, une partie de la présence française vient de collaborations : en comptage entier, la France a 52 modèles au lieu de 28,7 (×1,8), contre ×1,1 pour les États-Unis. Beaucoup viennent du laboratoire parisien de Facebook AI Research, attribué à la fois aux États-Unis et à la France.

### 2.4 Le retard en puissance de calcul

La puissance de calcul d'un modèle se mesure en FLOP, le nombre d'opérations faites pendant l'entraînement. On regarde d'abord le modèle le plus puissant de chaque année. Ensuite, pour chaque pays, on cherche en quelle année les États-Unis avaient déjà atteint son record : l'écart donne un retard en années.

In [ ]:
record = analyses.record_par_annee(epoch)
display(record.loc[record.index >= 2018].style.format({"record (FLOP)": "{:.1e}", "× par rapport à l'année précédente": "{:.1f}"}))
retard = annees_de_retard(epoch_pays).pivot(index="annee", columns="pays", values="retard_annees")
retard = retard.loc[retard.index >= 2018, [p for p in analyses.PAYS if p != "États-Unis"]]
courbes_par_pays(retard, "Retard sur les États-Unis, en années de puissance de calcul",
                 "Années de retard", fichier=FIGURES / "rapport_2_retard.png")
plt.show()

**Interprétation.** Le record de puissance de calcul a été multiplié par environ 100 000 entre 2018 (8,7 × 10²¹ FLOP) et 2026 (9,4 × 10²⁶ FLOP), avec jusqu'à ×18 en une seule année (2023). Depuis 2018, la France a entre 1 et 4 ans de retard sur les États-Unis. En 2026, son record correspond au niveau américain de 2023, soit 3 ans de retard, et l'écart ne se réduit pas. Ce calcul ne porte que sur les modèles dont la puissance de calcul est connue, et un seul modèle peut faire varier le retard d'un an.

## 3. Les développeurs publient-ils de moins en moins ?

### 3.1 Des informations qui disparaissent

In [ ]:
evolution = epoch.loc[epoch["annee"] >= 2015].groupby("annee")[list(INFOS_PUBLIABLES.values())].mean()
noms = {"publie_calcul": "Puissance de calcul", "publie_parametres": "Paramètres",
        "publie_donnees": "Taille des données", "publie_materiel": "Matériel", "publie_cout": "Coût"}
fig, ax = plt.subplots()
for (col, nom), c in zip(noms.items(), ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]):
    ax.plot(evolution.index, evolution[col], color=c, label=nom, linewidth=3 if col == "publie_calcul" else 1.6)
ax.axvline(2025.6, color="#8a8984", linestyle="--", linewidth=1)
ax.text(2025.5, 0.97, "AI Act", ha="right", fontsize=9, color="#52514e")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_ylim(0, 1)
ax.set_title("Part des modèles notables dont l'information est connue")
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.08), fontsize=9)
fig.tight_layout(); fig.savefig(FIGURES / "rapport_3_opacite.png"); plt.show()
evolution.loc[[2018, 2021, 2024, 2025, 2026]].style.format("{:.0%}")

**Interprétation.** H3 est confirmée. La part des modèles dont la puissance de calcul est connue atteint 78 % en 2021, puis tombe à 46 % en 2024 et 25 % en 2026. La taille des données d'entraînement suit la même tendance, et le coût n'est presque jamais connu (1 % en 2026). La baisse commence en 2022, quand les grands modèles deviennent des produits commerciaux. Pour les deux dernières années, une partie de la baisse vient aussi du fait qu'Epoch complète ses fiches avec du retard.

### 3.2 Information publiée ou estimée ?

In [ ]:
sources = (pd.crosstab(epoch.loc[epoch["annee"] >= 2018, "annee"],
                       epoch.loc[epoch["annee"] >= 2018, "source_calcul"], normalize="index")
           .reindex(columns=["calculée depuis des infos publiées", "estimation indirecte", "non renseignée"], fill_value=0))
sources.style.format("{:.0%}")

**Interprétation.** La baisse ne vient pas d'Epoch : les estimations indirectes restent rares (5 à 16 % des modèles selon les années depuis 2018). Ce qui baisse, c'est la part des modèles dont on peut calculer la puissance de calcul à partir de ce que publie le développeur : 67 % en 2021, 20 % en 2026.

### 3.3 Qui publie ?

Part des modèles dont la puissance de calcul est connue, par période, selon que les poids du modèle sont publiés (modèle ouvert, que tout le monde peut télécharger) ou non.

In [ ]:
publication = analyses.tableau_publication(epoch)
publication.style.format({"fermé": "{:.0%}", "poids ouverts": "{:.0%}"})

**Interprétation.** Jusqu'en 2020, modèles ouverts et fermés étaient aussi bien documentés (63 % et 64 %). Ensuite, la part des modèles fermés tombe à 55 % sur 2021-2023 puis à 18 % depuis 2024, alors que celle des modèles ouverts reste entre 64 % et 75 %. H4 est donc confirmée, mais seulement à partir de 2021. Chaque case compte plus de 120 modèles depuis 2021, donc ces écarts ne reposent pas sur quelques cas.

### 3.4 Deux mesures de la transparence qui ne disent pas la même chose

Stanford note ce que chaque développeur publie, sur une centaine d'indicateurs. Epoch indique ce qui est connu de ses modèles. Si les deux mesuraient la même chose, les points du graphique monteraient de gauche à droite.

In [ ]:
score = analyses.score_stanford(fmti, signataires)
comparaison = analyses.comparer_mesures(score, epoch)
nuage_de_points_annote(comparaison, "stanford", "calcul_publie_strict",
                       "Deux mesures de la transparence des développeurs",
                       "Score de transparence Stanford (édition 2025)",
                       "Modèles depuis 2024 dont la puissance de calcul\nest calculable à partir d'infos publiées",
                       fichier=FIGURES / "rapport_7_deux_mesures.png")
plt.show()
comparaison.round(2)

**Interprétation.** Ce n'est pas le cas. DeepSeek, Meta et Alibaba ont des modèles bien documentés (on peut calculer la puissance de calcul de 41 % à 90 % de leurs modèles récents), mais Stanford les classe parmi les moins transparents (26 % à 32 %). Anthropic et OpenAI sont dans la situation inverse : 46 % et 35 % chez Stanford, mais presque aucun modèle documenté (5 % et 6 %). Writer et AI21 Labs, en bas à droite, n'ont qu'un ou deux modèles dans Epoch, donc leur position compte peu. Les développeurs de modèles ouverts publient des fiches techniques, mais peu de choses sur leurs données, leurs usages ou leur gouvernance, que Stanford note aussi. H4 ne vaut donc que pour les informations techniques. Mistral, seul développeur français noté, tombe à 18 % chez Stanford en 2025 (55 % en 2024, avec une grille moins exigeante).

### 3.5 L'AI Act a-t-il changé quelque chose ?

On compare les modèles publiés depuis janvier 2024, avant et après le 2 août 2025, selon que le développeur a signé ou non le code de bonnes pratiques. C'est une simple comparaison, elle ne mesure pas un effet de l'AI Act.

In [ ]:
analyses.avant_apres_ai_act(epoch, signataires).style.format(
    {"modeles": "{:.0f}", "publie_calcul": "{:.0%}", "publie_donnees": "{:.0%}"})

**Interprétation.** H5 n'est pas confirmée. Les signataires (OpenAI, Anthropic, Google, Mistral…) publient beaucoup moins que les autres, avant comme après août 2025 : la puissance de calcul est connue pour 11 % de leurs modèles récents, contre 40 % pour les non-signataires. Les deux groupes baissent après l'AI Act. On ne peut pas pour autant dire que l'AI Act n'a pas d'effet : il y a encore peu de modèles après août 2025, Epoch met à jour ses fiches avec retard, et les signataires sont surtout de grands laboratoires fermés. De plus, l'AI Act demande un résumé des sources des données d'entraînement, pas leur taille, et ce résumé n'apparaît pas dans Epoch.

### 3.6 Le manque de transparence fausse-t-il la mesure du retard ?

Le retard calculé en 2.4 ne porte que sur les modèles dont la puissance de calcul est connue. Si ce sont surtout les modèles des leaders qui manquent, la comparaison est biaisée. On regarde donc, pour les modèles publiés depuis 2024, la part dont la puissance de calcul est connue, par pays puis pour les quatre grands laboratoires américains.

In [ ]:
par_pays = analyses.opacite_par_pays(epoch_pays)
# Pays avec au moins 5 modèles récents : en dessous, une part n'a pas de sens
barres_par_pays(par_pays.loc[par_pays["modeles"] >= 5, "part_calcul_connu"],
                "Modèles publiés depuis 2024 dont la puissance de calcul est connue",
                "Part des modèles du pays", fichier=FIGURES / "rapport_6_opacite_pays.png")
plt.show()
display(par_pays.round(2))
analyses.opacite_leaders(epoch).round(2)

**Interprétation.** Depuis 2024, la puissance de calcul est connue pour 27 % des modèles américains et 50 % des modèles chinois. L'écart vient des quatre grands laboratoires (OpenAI, Anthropic, Google DeepMind, xAI) : seulement 13 % de leurs 113 modèles récents, contre 49 % pour les autres développeurs. Or ce sont eux qui font les modèles de pointe (12 % de leurs modèles, contre 2 % pour les autres). Les modèles les plus puissants sont donc ceux qui manquent le plus, et le retard de la France est probablement sous-estimé. La France suit la même tendance : la puissance de calcul n'est connue que pour 14 % de ses modèles récents (Mistral). Le Royaume-Uni, l'Allemagne et le Canada ont moins de cinq modèles récents chacun, donc on ne commente pas leurs chiffres.

## 4. De quoi parle la recherche française en IA ?

On prend les résumés des 1 000 articles d'IA les plus cités depuis 2022 ayant au moins un auteur en France, et la même chose pour les États-Unis. On les nettoie avec spaCy comme dans le cours (lemmatisation, *stopwords*), en retirant aussi les mots qu'on retrouve dans tous les résumés scientifiques. Pour chaque mot, on calcule la part des articles qui le contiennent dans chaque pays. Le rapport entre les deux indique si le mot est plutôt « français » ou plutôt « américain ». On ne garde que les mots présents dans au moins 20 articles. Dans les nuages de mots, la taille d'un mot correspond à ce rapport.

In [ ]:
resumes = resumes.dropna(subset=["resume"])
resumes = resumes.loc[resumes["resume"].str.len() > 200]
resumes = resumes.loc[~resumes["titre"].fillna("").str.contains(r"\bquantum\b", case=False)]
resumes["pays"] = resumes["pays"].map({"fr": "France", "us": "États-Unis"})

nlp = spacy.load("en_core_web_sm")
resumes["tokens"] = nettoyer(resumes["resume"].tolist(), nlp)
comparaison_termes = comparer_presence(resumes.loc[resumes["pays"] == "France", "tokens"].tolist(),
                                       resumes.loc[resumes["pays"] == "États-Unis", "tokens"].tolist())
print(resumes.groupby("pays").size().to_dict())

**TF-IDF.** Comme dans le chapitre « bag of words » du cours, chaque résumé est un document. `TfidfVectorizer` donne à un mot un poids élevé s'il est fréquent dans un résumé mais rare dans l'ensemble des résumés. On fait ensuite la moyenne par pays.

In [ ]:
tfidf = tfidf_par_groupe(resumes["tokens"], resumes["pays"])
pd.DataFrame({p: [f"{t} ({v:.3f})" for t, v in serie.items()] for p, serie in tfidf.items()},
             index=range(1, 16))

**Interprétation.** Les premiers mots sont en partie les mêmes dans les deux pays (*data*, *learning*, *language*, *dataset*) : c'est le vocabulaire de base de l'IA. Mais aux États-Unis, *llm*, *prompt*, *reasoning* et *agent* sont dans le top 10, alors qu'en France on y trouve *speech* et *detection*, et *llm* n'arrive qu'en 15e position.

**Les mots propres à chaque pays.** Le TF-IDF fait surtout ressortir des mots communs aux deux corpus. Pour voir ce qui distingue chaque pays, on compare directement la part des articles qui contiennent chaque mot.

In [ ]:
extremes = pd.concat([comparaison_termes.head(12), comparaison_termes.tail(12)])
barres_rapport(extremes, "Les termes les plus « français » et les plus « américains »",
               fichier=FIGURES / "rapport_4_termes.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
cote_fr = comparaison_termes.head(30)
cote_us = comparaison_termes.tail(30)
nuage(dict(zip(cote_fr["terme"], cote_fr["rapport"])), axes[0], "Plus fréquents en France", COULEURS_PAYS["France"])
nuage(dict(zip(cote_us["terme"], 1 / cote_us["rapport"])), axes[1], "Plus fréquents aux États-Unis", COULEURS_PAYS["États-Unis"])
fig.tight_layout(); fig.savefig(FIGURES / "rapport_5_nuages.png"); plt.show()

**Interprétation.** Les articles français parlent beaucoup plus d'IA explicable : *xai* apparaît dans 10 fois plus d'articles qu'aux États-Unis (en proportion), *explainable* et *interpretation* 4 à 5 fois plus. On y trouve aussi la voix (*voice*), la détection d'anomalies, la santé (*cohort*) et des mots liés à l'Europe (*european*, *french*). Les articles américains parlent surtout des grands modèles de langage : *chatgpt*, *gpt*, *instruction*, *prompting*, le raisonnement étape par étape (*chain*, *thought*, *cot*) et le contournement des protections (*jailbreak*), 5 à 10 fois plus fréquents qu'en France. Cela rejoint la partie 2 : la France publie, mais peu sur les sujets où se joue la course aux modèles. Limite : les articles français sont beaucoup moins cités que les américains (médiane de 62 citations contre 508).

## Conclusion

La France est bien en retard, et ce n'est pas un effet de la transparence. Elle publie presque autant de recherche en IA par habitant que les États-Unis, mais ne produit pas de modèles de pointe. Sa présence repose sur Mistral AI et sur des collaborations, et elle a entre 1 et 4 ans de retard sur les États-Unis en puissance de calcul depuis 2018 (3 ans en 2026).

Les développeurs publient bien de moins en moins : la part des modèles dont la puissance de calcul est connue a été divisée par trois depuis 2021. Mais comme ce sont surtout les grands laboratoires américains qui ne publient pas (13 % de modèles documentés, contre 49 % pour les autres), le retard de la France est plutôt sous-estimé. Enfin, la transparence est difficile à mesurer : les modèles ouverts sont bien documentés techniquement mais pas sur leurs données et leurs usages, et l'AI Act n'a pas encore rendu les signataires plus transparents.

**Limites.** Epoch ne retient que les modèles « notables » et mesure ce qui est connu, pas forcément ce qui est publié par le développeur. OpenAlex donne le pays d'affiliation des auteurs, pas leur nationalité. Enfin, on a un peu plus d'un an de recul sur l'AI Act.

**Pour aller plus loin**, on pourrait suivre les résumés de données d'entraînement publiés au titre de l'AI Act, élargir la comparaison au reste de l'Europe, ou étudier le départ des chercheurs français vers les entreprises américaines.